# Lab 11: Long-Term Memory — the Maître d' Who Never Forgets a Guest

**Difficulty: Advanced | ~45 min | Requires Lab 9 (and Lab 10)**

Every guest who has to reintroduce themselves is a sign the restaurant forgot them. This lab builds the memory layer behind a one-table restaurant's maître d': an agent that remembers *Amara is allergic to cilantro*, *Amara loves tiramisu*, and *Amara's birthday is October 14* across separate sessions — then greets her by name in a brand-new thread with zero conversation history, avoids cilantro unprompted, and digs up the birthday menu when she asks *"what did you make me last year?"* Meanwhile Bob, a different guest, sees none of it. You will split agent memory into the three layers real systems use: a **checkpointer** (short-term, thread-scoped), a **store** (long-term, cross-thread), and **recall** (retrieval over stored facts).

**Cost:** ~15 OpenRouter calls on the free model per full run. No other APIs, no servers to host.

In [ ]:
!pip install -qU langchain==1.3.15 langchain-core==1.5.4 langchain-openai==1.4.3 langgraph==1.2.11 python-dotenv==1.2.2

**Step 2 — Imports, the key, and the measuring instrument.**

Same pinned stack and free OpenRouter model as Labs 5–10, same `.env`, same `UsageCapture` callback from Labs 8–9. The memory pieces are all `langgraph`: `Runtime` (the run-scoped object injected into nodes), `InMemoryStore` (the long-term store), `MemorySaver` (the thread checkpointer from Lab 9), and `dataclass` for the `Guest` context schema.

In [ ]:
import os, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path(".env"))
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain_core.callbacks import BaseCallbackHandler
from langchain.agents import create_agent
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver
from dataclasses import dataclass
from typing import Annotated, TypedDict

def model():
    return ChatOpenAI(base_url="https://openrouter.ai/api/v1",
                      api_key=os.environ["OPENROUTER_API_KEY"],
                      model="nvidia/nemotron-3-super-120b-a12b:free", temperature=0)

class UsageCapture(BaseCallbackHandler):
    def __init__(self): self.calls = []
    def on_llm_end(self, response, **kwargs):
        usage = (response.llm_output or {}).get("token_usage", {})
        self.calls.append(usage.get("prompt_tokens", 0))

**Step 3 — The store and the tools.**

One `InMemoryStore` serves the whole lab. `check_pantry` is a deterministic grounding tool (a fixed five-ingredient stock). `make_remember` is the *write path* of long-term memory: a factory that closes over `(store, guest_id)` and returns a `remember` tool. When the chef's model calls `remember(fact)`, the tool writes the fact into the guest's namespace — `put(namespace, key, value)` on a store is the entire long-term write API. The namespace tuple `("guests", guest_id, "facts")` is the isolation boundary: Bob's branch is a different branch of the tree.

In [ ]:
store = InMemoryStore()

@tool
def check_pantry(item: str) -> str:
    """Check whether an ingredient is in tonight's kitchen stock."""
    stock = {"thyme", "figs", "prosciutto", "saffron", "mascarpone"}
    return f"'{item}' is in stock" if item.lower() in stock else f"'{item}' is out of stock"

def make_remember(store, guest_id):
    @tool
    def remember(fact: str) -> str:
        """Persist one fact about this guest so every future session recalls it."""
        existing = [i.value["content"] for i in store.search(("guests", guest_id, "facts"))]
        if fact in existing:
            return "Already remembered."
        store.put(("guests", guest_id, "facts"), f"fact-{len(existing) + 1}", {"content": fact})
        return f"Remembered as fact {len(existing) + 1}."
    return remember


**Step 4 — Context schema, state, and the recall scorer.**

`Guest` is the `context_schema`: a dataclass the graph passes into every node, so `load_memory` and `chef` know *whose* evening this is. `MemoryState` is the graph state — one shared `messages` list merged with `add_messages` (same as Labs 9–10). `recall_score` is the retrieval stand-in: it counts overlapping words (length > 2) between the incoming message and a stored fact. In production you'd swap it for an embedding index; here it keeps recall deterministic and free.

In [ ]:
@dataclass
class Guest:
    guest_id: str

class MemoryState(TypedDict):
    messages: Annotated[list, add_messages]

def recall_score(query: str, fact: str) -> int:
    q = {w for w in query.lower().split() if len(w) > 2}
    f = {w for w in fact.lower().split() if len(w) > 2}
    return len(q & f)

**Step 5 — `load_memory`: memory as context.**

The *read path*. The model can't query the store, so the graph turns stored facts into a `SystemMessage` before every chef call. This node searches the guest's namespace, builds a dossier of every fact, and — when the guest's message overlaps stored facts — appends the top-2 as a recall line. Look at the signature: `runtime: Runtime[Guest]` is how LangGraph injects the run-scoped object carrying `runtime.store` and `runtime.context.guest_id`. No globals, no plumbing.

In [ ]:
def load_memory(state, runtime: Runtime[Guest]):
    guest = runtime.context.guest_id
    facts = runtime.store.search(("guests", guest, "facts"))
    dossier = "\n".join(f"- {i.value['content']}" for i in facts) or "(no facts yet — first visit)"
    hits = sorted(((recall_score(state["messages"][-1].content, i.value["content"]), i.value["content"])
                   for i in facts), reverse=True)[:2]
    recall = " | ".join(content for score, content in hits if score > 0)
    prompt = (f"You are the maître d' of a one-table restaurant. Your guest today is {guest}.\n"
              f"Facts you remember about {guest}:\n{dossier}\n")
    if recall:
        prompt += f"The guest's latest message matches these memories: {recall}\n"
    prompt += ("Greet warmly. Check the pantry with check_pantry when you suggest dishes. "
               "RULE: call remember for every NEW durable fact the guest shares about themselves - "
               "preferences, diet, allergies, plans. Never remember greetings or small talk, and never "
               "re-remember a fact you already hold. If there are no facts yet, remember what they told "
               "you, then ask one question.")
    return {"messages": [SystemMessage(content=prompt)]}


**Step 6 — `chef` and the graph.**

`chef_node` builds a fresh `create_agent` bound to `check_pantry` plus `remember` — created per-run with the *current* guest's id and the *current* store via `runtime`. The graph is a three-node pipeline compiled with `checkpointer=MemorySaver()` **and** `store=store`: the first gives thread-scoped conversation memory, the second gives cross-thread long-term memory. Both in one compile line is the whole lab in miniature.

In [ ]:
usage_calls, usage_counts = [], []

def chef_node(state, runtime: Runtime[Guest]):
    cap = UsageCapture()
    agent = create_agent(model=model(),
                         tools=[check_pantry, make_remember(runtime.store, runtime.context.guest_id)],
                         system_prompt="You are the maître d'. Serve the guest using the profile in your context.")
    result = agent.invoke({"messages": state["messages"]}, config={"callbacks": [cap]})
    usage_calls.append(cap.calls[0])
    usage_counts.append(len(cap.calls))
    return {"messages": result["messages"]}

app = (StateGraph(state_schema=MemoryState, context_schema=Guest)
       .add_node("load_memory", load_memory)
       .add_node("chef", chef_node)
       .add_edge(START, "load_memory")
       .add_edge("load_memory", "chef")
       .add_edge("chef", END)
       .compile(checkpointer=MemorySaver(), store=store))


**Step 7 — Evening 1: Amara fills the filing cabinet.**

The store starts empty. Amara volunteers four facts in one message; the chef should call `remember` for each, and `store.search` afterwards proves the writes landed. Three helpers earn their place here because evenings 2–4 reuse them: `run` (invoke the graph in a thread), `facts_of` (read a guest's dossier), `recall_line` (rank facts against a message).

In [ ]:
def facts_of(guest_id):
    return [i.value["content"] for i in store.search(("guests", guest_id, "facts"))]

def run(thread_id, guest_id, message):
    result = app.invoke({"messages": [("human", message)]},
                        config={"configurable": {"thread_id": thread_id}},
                        context=Guest(guest_id=guest_id))
    return str(result["messages"][-1].content)

def recall_line(query, guest_id):
    hits = sorted(((recall_score(query, i.value["content"]), i.value["content"])
                   for i in store.search(("guests", guest_id, "facts"))), reverse=True)[:2]
    return " | ".join(content for score, content in hits if score > 0)

print("EVENING 1 - Amara's first visit (thread: amara-1)")
answer = run("amara-1", "amara",
             "Good evening! I'm Amara. A few things about me: I'm allergic to cilantro. I love tiramisu. My birthday is October 14, and my grandmother always made saffron risotto for it.")
print("chef:", answer.replace("\n", " ")[:180])
print("store now holds:", facts_of("amara"))


EVENING 1 - Amara's first visit (thread: amara-1)
chef:   Good evening, Amara! Welcome to our restaurant. I hope you're having a lovely day.  I've taken note of a few things you shared: you're allergic to cilantro, you love tiramisu, yo
store now holds: ['Amara is allergic to cilantro.', 'Amara loves tiramisu.', "Amara's birthday is October 14.", "Amara's grandmother always made saffron risotto for her birthday."]


**Step 8 — Evening 2: a new thread with the old memory.**

Same guest, *different* `thread_id`. The conversation from evening 1 is gone — that is thread-scoped memory — but `load_memory` rebuilds the dossier from the store, so the chef greets Amara, remembers her birthday, and avoids cilantro. The print shows the dossier that was injected: memory as context, doing its job.

In [ ]:
print("EVENING 2 — Amara returns in a new thread (thread: amara-2)")
print("profile loaded:", "\n".join(f"- {f}" for f in facts_of("amara")))
answer = run("amara-2", "amara", "Hi, it's me again. What's on tonight?")
print("chef:", answer.replace("\n", " ")[:180])

EVENING 2 — Amara returns in a new thread (thread: amara-2)
profile loaded: - Amara is allergic to cilantro.
- Amara loves tiramisu.
- Amara's birthday is October 14.
- Amara's grandmother always made saffron risotto for her birthday.
chef:  Welcome back, Amara! It's a pleasure to see you again. I've checked our pantry tonight and found we have saffron and mascarpone in stock, which reminds me of your grandmother's sa


**Step 9 — Evening 2b: the same thread, a recall question.**

The conversation in this thread has already loaded Amara's dossier. Now she asks about last year's birthday. `load_memory` re-ranks the stored facts against *this* message and appends a recall line — the birthday fact — so the chef can answer from memory. This is retrieval over the store: the recall line is computed by `recall_score`, not copied by the model.

In [ ]:
print("EVENING 2b — same thread, a recall question")
query = "What did you make me for my birthday last year?"
print("recalled:", recall_line(query, "amara") or "(no match)")
answer = run("amara-2", "amara", query)
print("chef:", answer.replace("\n", " ")[:200])

EVENING 2b — same thread, a recall question
recalled: Amara's grandmother always made saffron risotto for her birthday. | Amara's birthday is October 14.
chef:  I don't have a specific record of what I prepared for your birthday last year in my memory. However, I do know that your grandmother always made saffron risotto for your birthday, and you have a part


**Step 10 — Evening 3 and the ledger.**

Bob gets a brand-new thread. His profile loads empty — no leakage from Amara — and his `remember` write lands in *his* namespace. The ledger closes with each guest's facts side by side, plus the decision-time token cost per chef call: run 2 and 2b cost more than run 1 because the dossier is now in their context. That token gap is the honest price of memory.

In [ ]:
print("EVENING 3 - Bob's first visit (thread: bob-1)")
answer = run("bob-1", "bob", "Hi, I'm Bob. First time here. I don't eat pork.")
print("profile loaded:", "\n".join(f"- {f}" for f in facts_of("bob")) or "(no facts yet - first visit)")
print("chef:", answer.replace("\n", " ")[:180])

print()
print("ledger")
print("  amara facts:", facts_of("amara"))
print("  bob   facts:", facts_of("bob"))
print("  decision-time tokens per chef call:", usage_calls)
print("  model calls per chef run:", usage_counts)


EVENING 3 - Bob's first visit (thread: bob-1)
profile loaded: - Bob does not eat pork
chef:   Hello Bob, welcome to our restaurant! It's a pleasure to have you here for your first visit.  May I ask if you have any other dietary preferences or restrictions I should be awar

ledger
  amara facts: ['Amara is allergic to cilantro.', 'Amara loves tiramisu.', "Amara's birthday is October 14.", "Amara's grandmother always made saffron risotto for her birthday."]
  bob   facts: ['Bob does not eat pork']
  decision-time tokens per chef call: [526, 527, 1225, 491]
  model calls per chef run: [6, 5, 1, 2]
